In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

In [4]:
def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(prob.value - (1-np.sum(a))*r_f+extra)
    return(prob.value,q.value,q_b.value)

In [5]:
def dual (sets,p,R,r,m,r_f,a):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R.dot(a))[i]-(1-sum(a))*r_f - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    obj= cp.Minimize(alpha + beta + gamma * (r-1) + z4 + z2)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,v.value,lbda.value,alpha.value,beta.value,gamma.value,t.value)

In [6]:
np.random.seed(10)
N=4
x=np.array([1,2,3])
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
psets

[[0],
 [1],
 [2],
 [3],
 [0, 1],
 [0, 2],
 [0, 3],
 [1, 2],
 [1, 3],
 [2, 3],
 [0, 1, 2],
 [0, 1, 3],
 [0, 2, 3],
 [1, 2, 3],
 [0, 1, 2, 3]]

In [10]:
a = np.array([ 30., -30., -30., -30.,  30.])
r = 1
m = 0.2
r_f = 0.01
c = 0.001
p = np.random.rand(4)
p = p/sum(p)
R = np.random.rand(4,5)*2-1
sets =psets

In [11]:
robustcheck(a,R,r,p,m,r_f)

74.52898644992649


(143.40540530146018,
 array([0.06359171, 0.06880225, 0.82063567, 0.04697037]),
 array([ 0.00000000e+00, -0.00000000e+00,  1.00000000e+00,  2.18889928e-12]))

In [12]:
dual (sets,p,R,r,m,r_f,a)

(74.52898645011683,
 array([[ 1.23067946e-10,  3.18769975e-11,  3.05321652e-12,
          3.13922554e-11],
        [ 3.20345419e-11,  1.75081321e-10, -0.00000000e+00,
         -0.00000000e+00],
        [-0.00000000e+00, -0.00000000e+00,  9.32812141e-10,
         -0.00000000e+00],
        [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
          1.67233384e-10],
        [ 1.28213200e-10,  1.20394336e-10, -0.00000000e+00,
         -0.00000000e+00],
        [ 1.21097047e-10, -0.00000000e+00,  4.26505878e-11,
         -0.00000000e+00],
        [ 1.27812839e-10, -0.00000000e+00, -0.00000000e+00,
          1.19519190e-10],
        [-0.00000000e+00,  1.12743090e-10,  4.24489144e-11,
         -0.00000000e+00],
        [-0.00000000e+00,  1.27910995e-10, -0.00000000e+00,
          1.27476397e-10],
        [-0.00000000e+00, -0.00000000e+00,  4.20356417e-11,
          1.11960196e-10],
        [ 1.08873567e-10,  1.00853039e-10,  2.44454195e-11,
         -0.00000000e+00],
        [ 1.15661060e-1

In [54]:
[probv,vv,lbdav,alphav,betav,gammav,tv]=dual (sets,p,R,r,m,r_f,a)
N = len(p)
M = len(sets)
cons1 =np.zeros(N)
cons2 = np.zeros(N)
z0 = 0
for j in range(M):
    z9 = -np.min(vv[j,sets[j]])*(1-m)+lbdav[j]
    z0 = z0 + max(z9,0)
for i in range(N):
    lbdsom = 0
    for j in range(M):
        if i in sets[j]:
            lbdsom = lbdsom + lbdav[j]
    cons1[i] = R.dot(a)[i] + betav + lbdsom
    cons2[i] = gammav * np.exp((-alphav+sum(vv[0:M:1,i]))/gammav)-tv[i]
print(cons1)
print(cons2)
print(-1+alphav+betav+gammav*r+sum(p*tv)+z0)

[ 5.70003067e-09 -5.38403810e-09  6.53539289e+00  1.70622106e+01]
[-8.10329546e-08 -2.98001260e-06 -3.94966149e-08 -2.06583066e-08]
[24.65265508]


array([4.38313766e-10, 1.34829779e-09, 5.62741161e-10, 4.45441049e-10])